# 02 — Avaliação da IA (M4)

Quão bem a classificação temática por embeddings acerta, comparada aos **temas oficiais** que a
própria Câmara atribui às proposições (`/proposicoes/{id}/temas`)? E o que os resumos dizem?

**Como medir.** Cada tema oficial é mapeado para um tema interno (`config/mapa_temas_oficiais.yaml`,
muitos-para-um). Uma proposição acerta quando o tema da IA está entre os temas internos esperados.
Temas oficiais sem correspondência (ex.: *Direito Constitucional*) não entram na conta.

**Cuidado com o vazamento.** O texto embutido de cada proposição inclui os temas oficiais (SPEC 10.3).
Por isso a acurácia na base inteira é otimista. O número honesto vem de uma **amostra sem vazamento**
(500 proposições embutidas de novo, só com tipo, ementa e palavras-chave), gravada por
`radar ai avaliar` em `ops.avaliacao_ia`. Este notebook não chama a OpenAI: só lê o banco.

In [ ]:
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from radar.ai.classifier import similaridades
from radar.ai.evaluation import base_avaliavel, carregar_mapa, top1
from radar.config import get_settings
from radar.db import get_engine

engine = get_engine()
limiar = get_settings().tema_limiar_minimo
pd.set_option("display.max_colwidth", 120)
with engine.connect() as conn:
    aval = (
        conn.execute(text("SELECT * FROM ops.avaliacao_ia ORDER BY id DESC LIMIT 1"))
        .mappings()
        .one()
    )
det = aval["detalhes"]
quando = aval["criado_em"].astimezone(ZoneInfo("America/Sao_Paulo"))
print(f"avaliação de {quando:%d/%m/%Y %H:%M} (Brasília) · limiar atual {limiar}")

## 1. Acurácia: base inteira × amostra sem vazamento

In [ ]:
sv = det["sem_vazamento"]
pd.DataFrame(
    [
        {
            "medida": "base inteira (com vazamento), top-1",
            "n": det["avaliadas"],
            "acurácia": det["acuracia_top1_com_vazamento"],
        },
        {"medida": "amostra sem vazamento, top-1", "n": sv["n"], "acurácia": sv["acuracia_top1"]},
    ]
).style.format({"acurácia": "{:.1%}"})

## 2. Calibração do limiar: acurácia × cobertura

Abaixo do limiar a proposição vai para **Outros** (conta como erro na acurácia geral). Critério: só
ficam abaixo do limiar as faixas de score em que a IA erra mais do que acerta. Depois de enriquecer a
descrição de *Tecnologia e Inovação* (ADR-026), até a faixa 0,30–0,35 acerta ~60%, e abaixo de 0,30 há
só 0,3% da base: limiar **0,30**.

In [ ]:
curva = pd.DataFrame(det["curva"]).set_index("limiar")
curva_sv = pd.DataFrame(sv["curva"]).set_index("limiar")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva_sv.index, curva_sv["acuracia"], "o-", label="acurácia geral (sem vazamento)")
ax.plot(
    curva_sv.index,
    curva_sv["acuracia_classificadas"],
    "s--",
    label="acurácia entre classificadas (sem vazamento)",
)
ax.plot(curva_sv.index, curva_sv["cobertura"], "^:", label="cobertura (sem vazamento)")
ax.axvline(limiar, color="gray", lw=1)
ax.axhline(0.70, color="red", lw=0.8, ls="--")
ax.set_xlabel("limiar mínimo de similaridade")
ax.set_ylim(0, 1.02)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Acurácia × cobertura por limiar")
plt.show()
curva_sv.loc[[0.275, 0.3, 0.325, 0.35, 0.4]].style.format("{:.1%}")

## 3. Distribuição dos scores (base inteira): acertos × erros

In [ ]:
base = base_avaliavel(engine)
df = top1(similaridades(engine, todas=True)).merge(
    base[["id_proposicao", "ementa", "temas_oficiais", "esperado"]], on="id_proposicao"
)
df["acerto"] = [t in e for t, e in zip(df["tema_top"], df["esperado"], strict=True)]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df.loc[df.acerto, "score"], bins=40, alpha=0.6, label="acertos")
ax.hist(df.loc[~df.acerto, "score"], bins=40, alpha=0.6, label="erros")
ax.axvline(limiar, color="gray", lw=1)
ax.set_xlabel("similaridade com o tema escolhido")
ax.legend()
plt.show()
print(f"{len(df)} proposições avaliáveis · acurácia top-1 {df.acerto.mean():.1%}")

## 4. Matriz de confusão

Linhas: tema interno esperado (do primeiro tema oficial mapeado). Colunas: tema da IA. A primeira
avaliação mostrou 377 PDLs de renovação de rádios comunitárias (tema oficial *Comunicações*) caindo em
*Meio Ambiente e Energia*; a descrição de *Tecnologia e Inovação* passou a citar radiodifusão e a
confusão sumiu. As fronteiras restantes refletem sobreposição real entre os temas.

In [ ]:
mapa = carregar_mapa()


def primeiro(temas):
    for t in temas or []:
        if mapa.get(t):
            return sorted(mapa[t])[0]


df["esperado_1"] = df["temas_oficiais"].map(primeiro)
matriz = pd.crosstab(df["esperado_1"], df["tema_top"], normalize="index")
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(matriz.to_numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(matriz.index)), matriz.index, fontsize=8)
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        v = matriz.to_numpy()[i, j]
        if v >= 0.05:
            ax.text(
                j,
                i,
                f"{v:.0%}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if v > 0.5 else "black",
            )
ax.set_xlabel("tema da IA")
ax.set_ylabel("tema esperado")
fig.colorbar(im, fraction=0.03)
plt.tight_layout()
plt.show()

## 5. Exemplos: 10 acertos e 10 erros

In [ ]:
cols = ["ementa", "temas_oficiais", "tema_top", "score"]
display(df[df.acerto].sample(10, random_state=1)[cols])
display(df[~df.acerto].sample(10, random_state=1)[cols])

## 6. Resumos executivos (gpt-4o-mini, prompt `resumo-v3`)

In [ ]:
with engine.connect() as conn:
    res = pd.DataFrame(
        conn.execute(
            text(
                "SELECT sigla_tipo, numero, ano, tema_ia, nivel_impacto, setores_afetados, resumo_executivo, relevancia_score "
                "FROM gold.fato_proposicao WHERE resumo_executivo IS NOT NULL"
            )
        )
        .mappings()
        .all()
    )
print(
    f"{len(res)} resumos · tamanho médio {res.resumo_executivo.str.len().mean():.0f} caracteres · "
    f"acima de 350: {(res.resumo_executivo.str.len() > 350).mean():.1%}"
)
display(res.nivel_impacto.value_counts(normalize=True).to_frame("fração").style.format("{:.1%}"))
display(res.explode("setores_afetados").setores_afetados.value_counts().to_frame("menções"))
display(pd.crosstab(res.tema_ia, res.nivel_impacto))

In [ ]:
# as 10 proposições mais relevantes
res.sort_values("relevancia_score", ascending=False).head(10)[
    [
        "sigla_tipo",
        "numero",
        "ano",
        "tema_ia",
        "nivel_impacto",
        "relevancia_score",
        "resumo_executivo",
    ]
]

## 7. Custos registrados (`ops.custos_ia`)

In [ ]:
with engine.connect() as conn:
    custos = pd.DataFrame(
        conn.execute(
            text(
                "SELECT id_execucao, operacao, modelo, sum(itens) AS itens, sum(tokens_entrada) AS tokens_entrada, "
                "sum(tokens_saida) AS tokens_saida, sum(custo_usd) AS custo_usd FROM ops.custos_ia "
                "GROUP BY 1, 2, 3 ORDER BY 1, 2"
            )
        )
        .mappings()
        .all()
    )
custos["custo_usd"] = custos["custo_usd"].astype(float)
display(custos)
print(f"total: US$ {custos.custo_usd.sum():.4f}")